In [8]:
if 'model' in globals():
    print("yay")

In [12]:

import torch
if torch.cuda.empty_cache():
    print("cleared")
    
gc.collect()

87

In [16]:
import torch
import gc
from typing import Optional, Tuple, Any
from transformers import AutoModelForCausalLM, AutoTokenizer

def print_gpu_memory():
    """Prints detailed GPU memory usage"""
    for i in range(torch.cuda.device_count()):
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1024**2
        allocated = torch.cuda.memory_allocated(i) / 1024**2
        reserved = torch.cuda.memory_reserved(i) / 1024**2
        free = total_memory - reserved
        print(f"GPU {i} Memory:")
        print(f"- Total: {total_memory:.0f}MB")
        print(f"- Reserved by cache: {reserved:.0f}MB")
        print(f"- Allocated: {allocated:.0f}MB")
        print(f"- Free: {free:.0f}MB")

def clear_gpu_memory(device_id: Optional[int] = None):
    """
    Aggressively cleans up GPU memory
    
    Args:
        device_id: If provided, only clear this GPU's memory
    """
    # First collect Python garbage to clean any dereferenced tensors
    gc.collect()
    
    # Validate device_id
    device_count = torch.cuda.device_count()
    if device_id is not None:
        if device_id >= device_count:
            raise ValueError(f"Invalid device_id {device_id}. Only {device_count} devices available.")
        devices = [device_id]
    else:
        devices = range(device_count)
    
    # Try to clear each device
    for device in devices:
        try:
            # First try to clear CUDA cache
            torch.cuda.empty_cache()
            
            # Then try device-specific operations
            with torch.cuda.device(f'cuda:{device}'):
                torch.cuda.empty_cache()
                if hasattr(torch.cuda.memory, 'empty_cache'):
                    torch.cuda.memory.empty_cache()
                torch.cuda.synchronize()
        except Exception as e:
            print(f"Warning: Error clearing GPU {device}: {str(e)}")
            
    # Delete all objects still in memory that have .cuda()
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj):
                if obj.is_cuda:
                    del obj
        except Exception:
            pass
    
    # One final garbage collection
    gc.collect()
    torch.cuda.empty_cache()

In [22]:
import requests
from lxml import etree as ET

PMCID = "PMC9314610"

URLS = [
    f"https://www.ebi.ac.uk/europepmc/webservices/rest/{PMCID}/fullTextXML",
    f"https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:{PMCID}&metadataPrefix=pmc",
]

DROP_HEADS = {
    "acknowledgements","acknowledgments","funding","funding information",
    "conflict of interest","competing interests","author contributions",
    "data availability","ethics","references","bibliography",
    "supplementary","appendix","correspondence"
}

def fetch_xml():
    last_err = None
    for url in URLS:
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            # Quick sanity check: must contain an <article> element
            if b"<article" in r.content:
                return ET.fromstring(r.content)
            # Some wrappers (OAI) put <article> deeper; still okay
            try:
                root = ET.fromstring(r.content)
                if root.xpath("//*[local-name()='article']"):
                    return root
            except ET.XMLSyntaxError as e:
                last_err = e
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to fetch XML for {PMCID}: {last_err}")

def norm_text(node):
    return " ".join(" ".join(node.itertext()).split())

def parse_title_abs_paras(root):
    # Find the article node regardless of namespaces/wrappers
    article_nodes = root.xpath("//*[local-name()='article']")
    if not article_nodes:
        return "", "", []
    article = article_nodes[0]

    # Title
    title_nodes = article.xpath(".//*[local-name()='article-title']")
    title = norm_text(title_nodes[0]) if title_nodes else ""

    # Abstract (join all parts)
    abs_nodes = article.xpath(".//*[local-name()='abstract']")
    abstract = norm_text(abs_nodes[0]) if abs_nodes else ""

    # Body → sections → paragraphs (skip unwanted heads)
    paras = []
    for sec in article.xpath(".//*[local-name()='body']//*[local-name()='sec']"):
        head = " ".join(sec.xpath(".//*[local-name()='title']/text()")).strip().lower()
        if any(bad in head for bad in DROP_HEADS):
            continue
        for p in sec.xpath(".//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    # Fallback: any <p> under body if secs were missed
    if not paras:
        for p in article.xpath(".//*[local-name()='body']//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    return title, abstract, paras

if __name__ == "__main__":
    root = fetch_xml()
    title, abstract, paras = parse_title_abs_paras(root)

    print("\nTITLE:\n", title, "\n")
    print("ABSTRACT (first 500 chars):\n", abstract[:500], "...\n")
    print("FIRST 3 PARAGRAPHS:\n")
    for i, p in enumerate(paras[:3], 1):
        print(f"[{i}] {p}\n")
    print(f"Total paragraphs found: {len(paras)}")



TITLE:
 Genotype–phenotype correlates in Joubert syndrome: A review 

ABSTRACT (first 500 chars):
 Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic varian ...

FIRST 3 PARAGRAPHS:

[1] Joubert syndrome (JS) is a rare congenital neurodevelopmental primary ciliopathy with a population‐based prevalence reaching 1.7 per 100,000 in the age range 0–19 years (Nuovo et al., 2020 ). First described by Dr Marie Joubert about 50 years ago (Joubert, Eisenring, Robb, & Andermann, 1969 ), JS is now diagnosed upon recognition of a pathognomonic malformation of th

In [7]:
# Test the improved GPU selection
print("Best GPU to use:", get_best_gpu())

NameError: name 'get_best_gpu' is not defined

In [17]:
# Print memory status before cleanup
print("Before cleanup:")
print_gpu_memory()

# Clear GPU 0 specifically
print("\nClearing GPU 0 memory...")
clear_gpu_memory(0)  # Changed from 1 to 0

# Print memory status after cleanup
print("\nAfter cleanup:")
print_gpu_memory()

Before cleanup:
GPU 0 Memory:
- Total: 12057MB
- Reserved by cache: 0MB
- Allocated: 0MB
- Free: 12057MB

Clearing GPU 0 memory...

After cleanup:
GPU 0 Memory:
- Total: 12057MB
- Reserved by cache: 0MB
- Allocated: 0MB
- Free: 12057MB


/home/guests/andreea_magureanu/.conda/envs/rare_dis/lib/python3.11/site-packages/torch/__init__.py:1125: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [18]:
# First ensure no existing model is in memory
try:
    del model
except:
    pass

# Force garbage collection and CUDA cache clearing
import gc
import torch
import os

# Set GPU 0 as the primary device
torch.cuda.set_device(0)
gc.collect()
torch.cuda.empty_cache()

# Check memory before loading
print("=== Memory Status Before Loading ===")
gpu_memory = get_gpu_memory_map()
for idx, gpu in enumerate(gpu_memory):
    print(f"\nGPU {idx}:")
    print(f"Total: {gpu['total']}MB")
    print(f"Used: {gpu['used']}MB")
    print(f"Free: {gpu['free']}MB")

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"

# Pick the best dtype for your GPU (bfloat16 if supported, else float16)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"\nUsing dtype: {dtype}")

# Load tokenizer first
tok = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=True,
    trust_remote_code=True,
)

# Ensure padding token exists
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

# Check memory again before loading model
print("\n=== Memory Status Before Model Load ===")
gpu_memory = get_gpu_memory_map()
for idx, gpu in enumerate(gpu_memory):
    print(f"\nGPU {idx}:")
    print(f"Free: {gpu['free']}MB")

# Load the model with memory monitoring
try:
    # First try loading with device map to GPU 0
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
        trust_remote_code=True,
        dtype=dtype,  # Using dtype instead of torch_dtype to avoid deprecation warning
        device_map={'': 0},  # Force everything to GPU 0
        max_memory={0: '8GiB'},  # Limit memory usage on GPU 0
        offload_folder="offload_folder",  # Enable disk offloading if needed
    )
    print("\nModel loaded successfully!")
except Exception as e:
    print(f"\nError loading model: {str(e)}")
    raise

# Final memory check
print("\n=== Final Memory Status ===")
gpu_memory = get_gpu_memory_map()
for idx, gpu in enumerate(gpu_memory):
    print(f"\nGPU {idx}:")
    print(f"Used: {gpu['used']}MB")
    print(f"Free: {gpu['free']}MB")

=== Memory Status Before Loading ===

GPU 0:
Total: 12288MB
Used: 310MB
Free: 11746MB

GPU 1:
Total: 12288MB
Used: 8522MB
Free: 3534MB

Using dtype: torch.bfloat16

=== Memory Status Before Model Load ===

GPU 0:
Free: 11744MB

GPU 1:
Free: 3534MB

=== Memory Status Before Model Load ===

GPU 0:
Free: 11744MB

GPU 1:
Free: 3534MB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Model loaded successfully!

=== Final Memory Status ===

GPU 0:
Used: 8532MB
Free: 3524MB

GPU 1:
Used: 8522MB
Free: 3534MB


In [19]:
# Check what processes are using GPU memory
import subprocess
import torch

def get_gpu_processes():
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory', '--format=csv,noheader'],
            encoding='utf-8'
        )
        print("=== GPU Processes ===")
        print("PID | Process Name | Memory Used")
        print("-" * 40)
        for line in result.strip().split('\n'):
            if line.strip():
                print(line)
    except Exception as e:
        print(f"Error getting GPU processes: {e}")

get_gpu_processes()

# Also check CUDA memory details
print("\n=== CUDA Memory Details ===")
torch.cuda.empty_cache()  # Try to clear cache first
for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}:")
    print(f"Memory allocated: {torch.cuda.memory_allocated(i) / 1024**2:.1f} MB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(i) / 1024**2:.1f} MB")
    print(f"Max memory allocated: {torch.cuda.max_memory_allocated(i) / 1024**2:.1f} MB")

=== GPU Processes ===
PID | Process Name | Memory Used
----------------------------------------
862314, /home/guests/andreea_magureanu/.conda/envs/rare_dis/bin/python, 8528 MiB
655696, python, 8518 MiB

=== CUDA Memory Details ===

GPU 0:
Memory allocated: 8201.8 MB
Memory reserved: 8222.0 MB
Max memory allocated: 8201.8 MB


In [2]:
# Cleanup GPU memory
import gc
import torch

# Move any existing models to CPU and clear them
if 'model' in globals():
    model.to('cpu')
    del model
if 'tokenizer' in globals():
    del tokenizer

# Clear all PyTorch caches
torch.cuda.empty_cache()
gc.collect()

# Force CUDA to compact memory
if hasattr(torch.cuda, 'memory_stats'):
    torch.cuda.memory_stats()
if hasattr(torch.cuda, 'reset_peak_memory_stats'):
    torch.cuda.reset_peak_memory_stats()

# Check memory after cleanup
print("\n=== Memory After Cleanup ===")
for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}:")
    print(f"Memory allocated: {torch.cuda.memory_allocated(i) / 1024**2:.1f} MB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(i) / 1024**2:.1f} MB")


=== Memory After Cleanup ===

GPU 0:
Memory allocated: 0.0 MB
Memory reserved: 0.0 MB


In [20]:
# Detailed GPU memory check
import torch
import subprocess
import psutil

def get_gpu_memory_map():
    """Get the current GPU memory usage."""
    result = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free,memory.total', '--format=csv,nounits,noheader'],
        encoding='utf-8'
    )
    
    gpu_memory = []
    for line in result.strip().split('\n'):
        used, free, total = map(int, line.split(','))
        gpu_memory.append({
            'total': total,
            'used': used,
            'free': free
        })
    return gpu_memory

def get_process_gpu_memory():
    """Get GPU memory used by current process."""
    process = psutil.Process()
    return {
        'memory_info': process.memory_info(),
        'gpu_memory': torch.cuda.memory_allocated() / 1024**2,
        'gpu_cached': torch.cuda.memory_reserved() / 1024**2
    }

print("=== System GPU Memory ===")
gpu_memory = get_gpu_memory_map()
for idx, gpu in enumerate(gpu_memory):
    print(f"\nGPU {idx}:")
    print(f"Total: {gpu['total']}MB")
    print(f"Used: {gpu['used']}MB")
    print(f"Free: {gpu['free']}MB")

print("\n=== Process Memory ===")
process_memory = get_process_gpu_memory()
print(f"Process RAM: {process_memory['memory_info'].rss / 1024**2:.0f}MB")
print(f"Process GPU Memory: {process_memory['gpu_memory']:.0f}MB")
print(f"Process GPU Cached: {process_memory['gpu_cached']:.0f}MB")

print("\n=== CUDA Memory Details ===")
print(f"Number of GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i} ({torch.cuda.get_device_name(i)}):")
    print(f"Allocated: {torch.cuda.memory_allocated(i) / 1024**2:.0f}MB")
    print(f"Cached: {torch.cuda.memory_reserved(i) / 1024**2:.0f}MB")
    
print("\n=== Current CUDA Device ===")
try:
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device properties: {torch.cuda.get_device_properties(torch.cuda.current_device())}")
except Exception as e:
    print(f"Error getting current device: {e}")

=== System GPU Memory ===

GPU 0:
Total: 12288MB
Used: 8532MB
Free: 3524MB

GPU 1:
Total: 12288MB
Used: 8522MB
Free: 3534MB

=== Process Memory ===
Process RAM: 1268MB
Process GPU Memory: 8202MB
Process GPU Cached: 8222MB

=== CUDA Memory Details ===
Number of GPUs: 1

GPU 0 (NVIDIA TITAN V):
Allocated: 8202MB
Cached: 8222MB

=== Current CUDA Device ===
Current device: 0
Device properties: _CudaDeviceProperties(name='NVIDIA TITAN V', major=7, minor=0, total_memory=12057MB, multi_processor_count=80, uuid=98f0c458-1ed0-e85f-02f4-488b0f234d0d, pci_bus_id=3, pci_device_id=0, pci_domain_id=0, L2_cache_size=4MB)


In [26]:
system = (
    "You are a medical IE agent. Output ONLY one JSON object with keys 'entities' and 'relations'. "
    "Each entity must be an object with: "
    "'name' (string), "
    "'type' (one of 'disease','gene','genotype','phenotype','treatment'), "
    "'aliases' (array of strings) which may be empty if not present in the text"
    " as for example common abbreviations or in parathenses after the first introduction of one of the entities"
    "related to rare disease, genes, genotypes, phenotype, or treatment"
    "Each relation must be an object with: 'subject','predicate','object'. "
    "If none, return empty arrays. No explanation, no extra text."
)
# system = (
#     "You are a medical IE agent. Output ONLY one JSON object with keys 'entities' and 'relations'. "
#     "Each entity must be an object with: "
#     "'name' (string), "
#     "'type' (one of 'disease','gene','genotype','phenotype','treatment'), "
#     "'aliases': an array of strings, may be empty."
#     "Aliases are ONLY allowed if they are explicitly in the TEXT."
#     "Valid alias cues:"
#     " - Parentheses after a term: e.g. 'Joubert syndrome (JS)' → alias 'JS'"
#     "  - Words like 'aka', 'also known as', 'abbreviated as', 'formerly called'"
#     "Do NOT invent synonyms or aliases not present in TEXT."
#     "Each relation must be an object with: 'subject','predicate','object'. "
#     "If none, return empty arrays. No explanation, no extra text."
# )
# system = (
#     "You are a medical IE agent. Output ONLY one JSON object with keys 'entities' . "
#     "Each entity must be an object with:  "
#     "'name' (string), "
#     "'aliases' (array of strings) which may be empty if not present in the text"
#     " as for example common abbreviations or in parathenses after the first introduction of one of the entities"
#     "related to rare disease"
# )


user   = (
    f"Title: {title}\nAbstract: {abstract}\n"
    "Identify diseases, genes, genotypes, phenotypes, treatments, and relations between them. "
    "Return only JSON. If none, return empty arrays."
)

messages = [
    {"role": "system", "content": system},
    {"role": "user",   "content": user},
]

In [27]:
import re, json
device = "cuda"
# Build chat prompt text (IMPORTANT: get a STRING, not tensors)
prompt_text = tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # mark where assistant should start
)
# n_prompt_tokens = len(tok(prompt_text)["input_ids"])
# MAX_CONTEXT = 8192
# max_new_tokens = MAX_CONTEXT - n_prompt_tokens - 50 
# print(max_new_tokens)

# Tokenize to a mapping (dict) for generate(**enc)
enc = tok(
    prompt_text,
    return_tensors="pt",
    add_special_tokens=False,
)

enc = {k: v.to(device) for k, v in enc.items()}

# Generate
out_ids = model.generate(
    **enc,
    max_new_tokens=2000,
    do_sample=False,                 # start deterministic
    eos_token_id=tok.eos_token_id,   # clean stop
    pad_token_id=tok.pad_token_id,   # silence warnings
)

# Decode only the new tokens (no prompt echo)
new_tokens = out_ids[0, enc["input_ids"].shape[-1]:]
out_text = tok.decode(new_tokens, skip_special_tokens=True).strip()

print("=== RAW MODEL OUTPUT ===")
print(out_text)

# Try to parse trailing JSON (optional, minimal)
m = re.search(r"\{[\s\S]*\}\s*$", out_text)
if m:
    try:
        data = json.loads(m.group(0))
        print("\n=== PARSED JSON ===")
        print(json.dumps(data, indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        pass

=== RAW MODEL OUTPUT ===
```json
{
  "entities": [
    {
      "name": "Joubert syndrome",
      "type": "disease",
      "aliases": []
    },
    {
      "name": "ciliopathy",
      "type": "disease",
      "aliases": []
    },
    {
      "name": "cerebellar malformation",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "brainstem malformation",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "molar tooth sign",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "liver fibrosis",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "renal involvement",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "retinal dystrophy",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "chronic kidney disease",
      "type": "phenotype",
      "aliases": []
    },
    {
      "name": "TMEM67",
      "type": "gene",
      "aliases": []
    },
    {
      "na

In [13]:

def extract_entities_from_output(json_output):
    """
    Extracts all entity names and their aliases from the model output
    Returns a dictionary of entities by type
    """
    entities_by_type = {
        'disease': set(),
        'gene': set(),
        'genotype': set(),
        'phenotype': set(),
        'treatment': set()
    }
    
    try:
        if isinstance(json_output, str):
            # Try to parse JSON from string
            data = json.loads(json_output)
        else:
            data = json_output
            
        if 'entities' in data:
            for entity in data['entities']:
                # Add the main name
                if 'type' in entity and 'name' in entity:
                    entities_by_type[entity['type']].add(entity['name'])
                    
                # Add all aliases
                if 'aliases' in entity and entity['aliases']:
                    entities_by_type[entity['type']].update(entity['aliases'])
    
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Error processing entities: {e}")
        
    return entities_by_type

def create_context_prompt(previous_entities):
    """
    Creates a context prompt from previously found entities
    """
    context_lines = []
    for entity_type, entities in previous_entities.items():
        if entities:
            entities_str = ", ".join(sorted(entities))
            context_lines.append(f"Previously mentioned {entity_type}s: {entities_str}")
    
    return "\n".join(context_lines)

In [14]:

all_entities = {
    'disease': set(),
    'gene': set(),
    'genotype': set(),
    'phenotype': set(),
    'treatment': set()
}


initial_text = f"Title: {title}\nAbstract: {abstract}"
current_context = ""

def process_text_with_context(text, context=""):

    if context:
        user_prompt = f"{text}\n\nContext:\n{context}\n\nIdentify diseases, genes, genotypes, phenotypes, treatments, and relations between them. Consider the provided context of previously mentioned entities. Return only JSON. If none, return empty arrays."
    else:
        user_prompt = f"{text}\nIdentify diseases, genes, genotypes, phenotypes, treatments, and relations between them. Return only JSON. If none, return empty arrays."
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_prompt},
    ]
    
  
    prompt_text = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    

    enc = tok(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    

    out_ids = model.generate(
        **enc,
        max_new_tokens=2000,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )
    

    new_tokens = out_ids[0, enc["input_ids"].shape[-1]:]
    out_text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    
 
    m = re.search(r"\{[\s\S]*\}\s*$", out_text)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            return {"entities": [], "relations": []}
    return {"entities": [], "relations": []}


print("Processing title and abstract...")
result = process_text_with_context(initial_text)
new_entities = extract_entities_from_output(result)
for entity_type in all_entities:
    all_entities[entity_type].update(new_entities[entity_type])


current_context = create_context_prompt(all_entities)
print("\nInitial entities found:")
print(json.dumps(all_entities, indent=2, default=list))


for i, para in enumerate(paras[:3]):  
    print(f"\nProcessing paragraph {i+1}...")
    result = process_text_with_context(para, current_context)
    

    new_entities = extract_entities_from_output(result)
    for entity_type in all_entities:
        all_entities[entity_type].update(new_entities[entity_type])
    

    current_context = create_context_prompt(all_entities)
    
    print(f"Entities after paragraph {i+1}:")
    print(json.dumps(all_entities, indent=2, default=list))

print("\nFinal set of unique entities:")
for entity_type, entities in all_entities.items():
    if entities:
        print(f"\n{entity_type.capitalize()}:")
        for entity in sorted(entities):
            print(f"- {entity}")

NameError: name 'title' is not defined